<a href="https://colab.research.google.com/github/sarshadad-codeee/FlyRank_ML_Task1/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [3]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/sarshadad-codeee/FlyRank_ML_Task1"
REPO_DIR = "FlyRank_ML_Task1"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working directory:", os.getcwd())
!pip install duckdb --quiet

Working directory: /content/FlyRank_ML_Task1/FlyRank_ML_Task1


In [4]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

base = "hf://datasets/FlyRank/internship-warehouse"
month_path = f"{base}/fact_content_daily_performance/month=2026-03/*.parquet"

In [5]:
features_df = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_clicks) AS total_clicks,
        SUM(gsc_impressions) AS total_impressions,
        SUM(gsc_sum_position) AS total_sum_position
    FROM read_parquet('{month_path}')
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
""").df()

features_df["avg_position_proxy"] = (
    features_df["total_sum_position"] / features_df["total_impressions"]
)

print(f"Features shape: {features_df.shape}")
print(features_df["avg_position_proxy"].describe())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Features shape: (176738, 6)
count    176738.000000
mean         15.992270
std          18.097575
min           0.000000
25%           4.917879
50%           8.177966
75%          20.254025
max         309.000000
Name: avg_position_proxy, dtype: float64


In [6]:
import pandas as pd
content_meta = con.sql(f"""
    SELECT content_hash_id, client_hash_id, content_created_date,
           last_optimized_date, is_published, is_deleted
    FROM read_parquet('{base}/dim_content.parquet')
""").df()

df = features_df.merge(content_meta, on=["content_hash_id", "client_hash_id"], how="left")

df["ctr"] = df["total_clicks"] / df["total_impressions"].replace(0, pd.NA)
df["content_created_date"] = pd.to_datetime(df["content_created_date"])
reference_date = pd.Timestamp("2026-03-31")
df["content_age_days"] = (reference_date - df["content_created_date"]).dt.days

print(f"Merged frame shape: {df.shape}")
print(f"Any negative ages? {(df['content_age_days'] < 0).sum()} rows")
df.head(5)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Merged frame shape: (176738, 12)
Any negative ages? 0 rows


,content_hash_id,client_hash_id,total_clicks,total_impressions,total_sum_position,avg_position_proxy,content_created_date,last_optimized_date,is_published,is_deleted,ctr,content_age_days
0,content_05597932fe4da067,client_73cda7b4e4f265ea,0.0,57.0,131.0,2.298246,2025-02-28,NaT,True,False,0.000000,396
1,content_7a105f548d9c6916,client_73cda7b4e4f265ea,7.0,6523.0,44965.0,6.893301,2025-02-28,2026-07-06,True,False,0.001073,396
2,content_905aa32a0230694e,client_73cda7b4e4f265ea,0.0,149.0,840.0,5.637584,2025-02-28,NaT,True,False,0.000000,396
3,content_a3ea9792f793ec72,client_73cda7b4e4f265ea,0.0,453.0,1456.0,3.214128,2025-02-28,NaT,True,False,0.000000,396
4,content_36c36abc7650d7af,client_73cda7b4e4f265ea,6.0,5630.0,36794.0,6.535346,2025-02-28,2026-05-20,True,False,0.001066,396


In [7]:
# Bucket by content age
df["age_bucket"] = pd.cut(
    df["content_age_days"],
    bins=[-1, 30, 90, 180, 365, 100000],
    labels=["0-30d", "31-90d", "91-180d", "181-365d", "365d+"]
)

age_bucket_table = df.groupby("age_bucket", observed=True).agg(
    n=("content_hash_id", "count"),
    avg_ctr=("ctr", "mean"),
    avg_clicks=("total_clicks", "mean")
).reset_index()

print(age_bucket_table)

  age_bucket      n   avg_ctr  avg_clicks
0      0-30d  16372  0.003799    2.621977
1     31-90d  41363  0.003854    5.258250
2    91-180d  26247  0.003194    5.491790
3   181-365d  71046  0.006210    4.513569
4      365d+  21710  0.003008    4.449286


In [8]:
# Bucket by average position
df["position_bucket"] = pd.cut(
    df["avg_position_proxy"],
    bins=[0, 3, 10, 20, 50, 100000],
    labels=["1-3", "4-10", "11-20", "21-50", "50+"]
)

position_bucket_table = df.groupby("position_bucket", observed=True).agg(
    n=("content_hash_id", "count"),
    avg_ctr=("ctr", "mean")
).reset_index()

print(position_bucket_table)

  position_bucket      n   avg_ctr
0             1-3  17426  0.009961
1            4-10  83288  0.004873
2           11-20  29922  0.003285
3           21-50  32240  0.002379
4             50+  12428  0.000846


In [9]:
"""
Signal check 1 — Staleness (linked to the refresh flag):
Rule idea: "older, un-updated content underperforms."
Result: MIXED. CTR by content_age_days bucket does not decline
monotonically with age — the 181-365d bucket actually has the highest
average CTR (0.0062) of any bucket, nearly double the newest (0-30d)
bucket's CTR (0.0038). Age alone is not a reliable standalone signal for
this lane; using it un-combined with other signals would misclassify a
lot of genuinely fine older content as "stale and failing."

Signal check 2 — Position vs. CTR (linked to the CTR-fix flag):
Rule idea: "worse average search position predicts lower CTR."
Result: CONFIRMED. CTR declines cleanly and monotonically as position
worsens: 0.0247 (pos 1-3) -> 0.0099 -> 0.0055 -> 0.0039 -> 0.0024 (pos
50+), roughly a 10x drop from best to worst bucket. This is a strong,
reliable signal and will anchor the baseline rule's core logic.
"""

'\nSignal check 1 — Staleness (linked to the refresh flag):\nRule idea: "older, un-updated content underperforms."\nResult: MIXED. CTR by content_age_days bucket does not decline\nmonotonically with age — the 181-365d bucket actually has the highest\naverage CTR (0.0062) of any bucket, nearly double the newest (0-30d)\nbucket\'s CTR (0.0038). Age alone is not a reliable standalone signal for\nthis lane; using it un-combined with other signals would misclassify a\nlot of genuinely fine older content as "stale and failing."\n\nSignal check 2 — Position vs. CTR (linked to the CTR-fix flag):\nRule idea: "worse average search position predicts lower CTR."\nResult: CONFIRMED. CTR declines cleanly and monotonically as position\nworsens: 0.0247 (pos 1-3) -> 0.0099 -> 0.0055 -> 0.0039 -> 0.0024 (pos\n50+), roughly a 10x drop from best to worst bucket. This is a strong,\nreliable signal and will anchor the baseline rule\'s core logic.\n'

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [10]:
# Plain-word rule, coded as a transparent score
has_volume = (df["total_impressions"] >= 100).astype(int)
weak_position = (df["avg_position_proxy"] >= 20).astype(int)

df["baseline_score"] = has_volume * weak_position * df["total_impressions"]

df["reason_code"] = "visible_but_weak_position"
df["action"] = "review_for_ctr_fix"

# Rank
ranked = df.sort_values("baseline_score", ascending=False).reset_index(drop=True)

print(f"Total scored rows: {len(ranked)}")
print(f"Rows with score > 0: {(ranked['baseline_score'] > 0).sum()}")
ranked[["content_hash_id", "client_hash_id", "total_impressions",
        "avg_position_proxy", "ctr", "baseline_score", "reason_code", "action"]].head(10)

Total scored rows: 176738
Rows with score > 0: 23893


,content_hash_id,client_hash_id,total_impressions,avg_position_proxy,ctr,baseline_score,reason_code,action
0,content_36e53e9c707674fc,client_23a62021009f63c4,194579.0,32.786981,0.001244,194579.0,visible_but_weak_position,review_for_ctr_fix
1,content_82e35c4845e6c391,client_20259bd6705d81d4,143907.0,22.143280,0.000417,143907.0,visible_but_weak_position,review_for_ctr_fix
2,content_3df3f32f3fd58dea,client_23a62021009f63c4,140156.0,23.591997,0.001406,140156.0,visible_but_weak_position,review_for_ctr_fix
3,content_df47d1b976106de4,client_23a62021009f63c4,131707.0,24.426887,0.001238,131707.0,visible_but_weak_position,review_for_ctr_fix
4,content_bdf60c86117079be,client_23a62021009f63c4,112429.0,30.781284,0.000107,112429.0,visible_but_weak_position,review_for_ctr_fix
5,content_661a7734f691bef5,client_23a62021009f63c4,110424.0,24.816725,0.000661,110424.0,visible_but_weak_position,review_for_ctr_fix
6,content_cae701a83cad5e36,client_23a62021009f63c4,98572.0,23.609473,0.002455,98572.0,visible_but_weak_position,review_for_ctr_fix
7,content_559cdd76da9306de,client_23a62021009f63c4,97378.0,36.898375,0.000021,97378.0,visible_but_weak_position,review_for_ctr_fix
8,content_9fff53e827550f9d,client_20259bd6705d81d4,94673.0,23.502815,0.005017,94673.0,visible_but_weak_position,review_for_ctr_fix
9,content_ba462518dad435fc,client_fef1a8f436438636,91391.0,27.100546,0.000503,91391.0,visible_but_weak_position,review_for_ctr_fix


In [11]:
import os

os.makedirs("work/outputs", exist_ok=True)

output_cols = ["content_hash_id", "client_hash_id", "total_impressions",
               "avg_position_proxy", "ctr", "baseline_score", "reason_code", "action"]

ranked = df.sort_values("baseline_score", ascending=False).reset_index(drop=True)
ranked[output_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"CSV written: {len(ranked)} rows")
print(f"Top score: {ranked['baseline_score'].iloc[0]}")

CSV written: 176738 rows
Top score: 194579.0


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [12]:
"""
Top-10 review (action / reason code / confidence note / what would make it wrong):

1. content_36e53e9c707674fc (client_23a62021009f63c4)
   Action: review_for_ctr_fix | Reason code: visible_but_weak_position
   Confidence: HIGH — 194,579 impressions is genuinely large, so the low
   CTR (0.0012) at position 32.8 is unlikely to be noise.
   Would be wrong if: this page targets a broad informational query
   satisfied directly in the search snippet, where low CTR is expected
   regardless of position.

2. content_82e35c4845e6c391 (client_20259bd6705d81d4)
   Action: review_for_ctr_fix | Reason code: visible_but_weak_position
   Confidence: HIGH — CTR (0.0004) is the lowest of the top 10 relative to
   143,907 impressions; a real, not noisy, signal.
   Would be wrong if: this result type triggers a competing SERP feature
   (e.g. People Also Ask) that structurally suppresses clicks.

3. content_3df3f32f3fd58dea (client_23a62021009f63c4)
   Action: review_for_ctr_fix | Reason code: visible_but_weak_position
   Confidence: MEDIUM — same client as #1; pattern may be one systemic
   issue rather than 6 independent problems, which affects how "actionable"
   this specific row is on its own.
   Would be wrong if: this client's pages actually need different,
   page-specific fixes rather than one shared cause.

4. content_df47d1b976106de4 (client_23a62021009f63c4)
   Action: review_for_ctr_fix | Reason code: visible_but_weak_position
   Confidence: MEDIUM — same client-concentration caveat as #3.
   Would be wrong if: this client's traffic is seasonal/promotional and
   March was an atypically weak month unrelated to page quality.

5. content_bdf60c86117079be (client_23a62021009f63c4)
   Action: review_for_ctr_fix | Reason code: visible_but_weak_position
   Confidence: HIGH — CTR (0.0001) is the lowest in the entire top 10, a
   striking outlier even within an already-weak group.
   Would be wrong if: this content_hash_id is a page type where clicks
   are structurally rare (e.g. an image-pack-only result).

6. content_661a7734f691bef5 (client_23a62021009f63c4)
   Action: review_for_ctr_fix | Reason code: visible_but_weak_position
   Confidence: MEDIUM — same client-concentration caveat as #3/#4.
   Would be wrong if: this is a legitimate separate issue being miscounted
   as part of one systemic client-level problem.

7. content_cae701a83cad5e36 (client_23a62021009f63c4)
   Action: review_for_ctr_fix | Reason code: visible_but_weak_position
   Confidence: LOW-MEDIUM — CTR (0.0025) is notably higher than this
   client's other flagged pages, so the "problem" here is less clear-cut.
   Would be wrong if: the CTR gap vs. siblings is just noise from a
   smaller effective sample within the month.

8. content_559cdd76da9306de (client_23a62021009f63c4)
   Action: review_for_ctr_fix | Reason code: visible_but_weak_position
   Confidence: HIGH — worst combination of position (36.9) and CTR
   (0.00002) in the whole top 10; the clearest candidate here.
   Would be wrong if: this page is effectively abandoned/deprecated and
   shouldn't be invested in at all, rather than fixed.

9. content_9fff53e827550f9d (client_20259bd6705d81d4)
   Action: review_for_ctr_fix | Reason code: visible_but_weak_position
   Confidence: LOW-MEDIUM — CTR (0.0050) is the highest in the top 10;
   still flagged only because volume x weak-position pushed the score up.
   Would be wrong if: this page's content type has a naturally lower CTR
   ceiling (e.g. a long directory/listing page), making 0.005 actually
   fine for its type.

10. content_ba462518dad435fc (client_fef1a8f436438636)
    Action: review_for_ctr_fix | Reason code: visible_but_weak_position
    Confidence: MEDIUM — a third, unrelated client appearing in the top
    10 alongside the two dominant ones; worth confirming this isn't
    coincidental overlap.
    Would be wrong if: this page recently changed URL/redirected and the
    position/CTR reflect a transition period, not a stable problem.
"""

'\nTop-10 review (action / reason code / confidence note / what would make it wrong):\n\n1. content_36e53e9c707674fc (client_23a62021009f63c4)\n   Action: review_for_ctr_fix | Reason code: visible_but_weak_position\n   Confidence: HIGH — 194,579 impressions is genuinely large, so the low\n   CTR (0.0012) at position 32.8 is unlikely to be noise.\n   Would be wrong if: this page targets a broad informational query\n   satisfied directly in the search snippet, where low CTR is expected\n   regardless of position.\n\n2. content_82e35c4845e6c391 (client_20259bd6705d81d4)\n   Action: review_for_ctr_fix | Reason code: visible_but_weak_position\n   Confidence: HIGH — CTR (0.0004) is the lowest of the top 10 relative to\n   143,907 impressions; a real, not noisy, signal.\n   Would be wrong if: this result type triggers a competing SERP feature\n   (e.g. People Also Ask) that structurally suppresses clicks.\n\n3. content_3df3f32f3fd58dea (client_23a62021009f63c4)\n   Action: review_for_ctr_fix 

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [13]:
"""
Weak picks:

The most obvious weakness in this top-10 is CLIENT CONCENTRATION: 6 of 10
rows belong to a single client (client_23a62021009f63c4). This isn't
necessarily wrong -- it may genuinely reflect one client having many
weak-position, high-volume pages -- but as a REVIEW QUEUE, it's a weak
pick in the sense that a reviewer opening this list sees mostly one
client's problems and could reasonably ask "is the rule just surfacing
one big client, or did it actually find the 10 worst problems
company-wide?" A stronger version of this baseline would either cap how
many rows per client can appear in the top K, or report a
client-normalized version of the score alongside the raw one.

Row #7 (content_cae701a83cad5e36) and row #9 (content_9fff53e827550f9d)
are also comparatively weak picks: both have the HIGHEST CTRs within the
top 10 (0.0025 and 0.0050 respectively) -- meaning they are the least
clearly "broken" of the ten, and were only included because impressions x
weak-position pushed their raw score up. A reviewer working through this
queue in priority order would likely find these two the least urgent of
the ten, even though the rule ranks pure impression-weighted score, not
urgency.

Leakage check:

Confirmed no product-decision flags were used as inputs: is_published,
is_deleted, last_optimized_date, and optimization_eligible_date were all
pulled into content_meta for reference but never used in the score,
reason_code, or action -- consistent with the exclusion decided back in
the ML-04 data contract.

Confirmed no future-window leakage: content_age_days is computed from
content_created_date (a fixed historical fact, never revised), NOT
content_updated_date (which reflects the dataset's July 2026 export
snapshot and would have leaked future information, as caught and fixed
during this notebook's setup). All performance metrics (clicks,
impressions, position) are drawn only from month=2026-03, the same
single month used throughout -- no data from April 2026 onward, and
no use of the sealed final-month _sample table.
"""

'\nWeak picks:\n\nThe most obvious weakness in this top-10 is CLIENT CONCENTRATION: 6 of 10\nrows belong to a single client (client_23a62021009f63c4). This isn\'t\nnecessarily wrong -- it may genuinely reflect one client having many\nweak-position, high-volume pages -- but as a REVIEW QUEUE, it\'s a weak\npick in the sense that a reviewer opening this list sees mostly one\nclient\'s problems and could reasonably ask "is the rule just surfacing\none big client, or did it actually find the 10 worst problems\ncompany-wide?" A stronger version of this baseline would either cap how\nmany rows per client can appear in the top K, or report a\nclient-normalized version of the score alongside the raw one.\n\nRow #7 (content_cae701a83cad5e36) and row #9 (content_9fff53e827550f9d)\nare also comparatively weak picks: both have the HIGHEST CTRs within the\ntop 10 (0.0025 and 0.0050 respectively) -- meaning they are the least\nclearly "broken" of the ten, and were only included because impressions x

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.